In [2]:
import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from tqdm import tqdm

df_raw = pd.read_csv("edge_features_abs.csv")
df_z = pd.read_csv("edge_features_z.csv")

feature_names = df_raw.drop(columns=["Subject", "Label"]).columns

X_raw = df_raw.drop(columns=["Subject", "Label"]).values
X_z = df_z.drop(columns=["Subject", "Label"]).values

y = LabelEncoder().fit_transform(df_raw["Label"])

K = 30

In [12]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

def run_one_bootstrap(X_raw, X_z, y, seed, K=30, max_features=5):
    """
    One participant-level stratified bootstrap iteration.
    Train on bootstrap sample, test on OOB subjects.
    Feature selection runs on training subjects only (no leakage).
    """
    rng = np.random.RandomState(seed)
    n = len(y)

    # Stratified bootstrap: sample within each class separately
    ad_idx = np.where(y == 1)[0]
    cn_idx = np.where(y == 0)[0]

    ad_boot = rng.choice(ad_idx, size=len(ad_idx), replace=True)
    cn_boot = rng.choice(cn_idx, size=len(cn_idx), replace=True)
    train_idx = np.concatenate([ad_boot, cn_boot])

    # OOB = subjects whose original index never appeared in the bootstrap draw
    oob_mask = np.ones(n, dtype=bool)
    oob_mask[np.unique(train_idx)] = False          # unique original indices
    test_idx = np.where(oob_mask)[0]

    # Skip degenerate draws (very rare: OOB has only one class)
    if len(np.unique(y[test_idx])) < 2:
        return None

    X_train_raw = X_raw[train_idx]
    X_train_z   = X_z[train_idx]
    X_test_z    = X_z[test_idx]       # evaluated on Fisher-Z values
    y_train     = y[train_idx]
    y_test      = y[test_idx]

    # Feature selection on training subjects only
    mwu_scores = np.array([
        -np.log10(mannwhitneyu(
            X_train_raw[y_train == 0, f],
            X_train_raw[y_train == 1, f],
            alternative="two-sided"
        ).pvalue + 1e-10)
        for f in range(X_train_raw.shape[1])
    ])
    top_features = np.argsort(mwu_scores)[::-1][:K]

    # Train and evaluate
    rf = RandomForestClassifier(n_estimators=90,
                                max_features=max_features,
                                random_state=42)
    rf.fit(X_train_z[:, top_features], y_train)

    pred = rf.predict(X_test_z[:, top_features])
    prob = rf.predict_proba(X_test_z[:, top_features])[:, 1]

    return {
        "accuracy":  accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall":    recall_score(y_test, pred, zero_division=0),
        "f1":        f1_score(y_test, pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_test, prob),
    }


# Run bootstrap
N_BOOTSTRAPS = 1000
results = []

for b in tqdm(range(N_BOOTSTRAPS), desc="Bootstraps"):
    res = run_one_bootstrap(X_raw, X_z, y, seed=b, K=30, max_features=5)
    if res is not None:
        res["bootstrap"] = b + 1
        results.append(res)

df_boot = pd.DataFrame(results)

# Report CI
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    vals = df_boot[metric].values
    ci = np.percentile(vals, [2.5, 97.5])
    print(f"{metric:12s}: mean={np.mean(vals):.4f}  "
          f"95% CI [{ci[0]:.4f}, {ci[1]:.4f}]")

Bootstraps: 100%|██████████| 1000/1000 [04:25<00:00,  3.77it/s]


accuracy    : mean=0.5794  95% CI [0.4091, 0.7392]
precision   : mean=0.6286  95% CI [0.4375, 0.8182]
recall      : mean=0.6572  95% CI [0.3750, 0.9231]
f1          : mean=0.6329  95% CI [0.4348, 0.7857]
roc_auc     : mean=0.6107  95% CI [0.3958, 0.7974]


In [ ]:
def run_bootstrap_with_edges(X_raw, X_z, y, feature_names,
                              seed, K=30, max_features=5):
    rng = np.random.RandomState(seed)
    n = len(y)

    # Stratified bootstrap
    ad_idx = np.where(y == 1)[0]
    cn_idx = np.where(y == 0)[0]
    train_idx = np.concatenate([
        rng.choice(ad_idx, size=len(ad_idx), replace=True),
        rng.choice(cn_idx, size=len(cn_idx), replace=True)
    ])

    oob_mask = np.ones(n, dtype=bool)
    oob_mask[np.unique(train_idx)] = False
    test_idx = np.where(oob_mask)[0]

    if len(np.unique(y[test_idx])) < 2:
        return None

    X_train_raw = X_raw[train_idx]
    X_train_z   = X_z[train_idx]
    X_test_z    = X_z[test_idx]
    y_train     = y[train_idx]
    y_test      = y[test_idx]

    # MWU feature selection on training subjects only
    mwu_scores = np.array([
        -np.log10(mannwhitneyu(
            X_train_raw[y_train == 0, f],
            X_train_raw[y_train == 1, f],
            alternative="two-sided"
        ).pvalue + 1e-10)
        for f in range(X_train_raw.shape[1])
    ])
    top_features = np.argsort(mwu_scores)[::-1][:K]

    # Train RF
    rf = RandomForestClassifier(n_estimators=90,
                                max_features=max_features,
                                random_state=42)
    rf.fit(X_train_z[:, top_features], y_train)

    pred = rf.predict(X_test_z[:, top_features])
    prob = rf.predict_proba(X_test_z[:, top_features])[:, 1]

    # Track both: which edges were selected AND their Gini importance
    edge_selected = np.zeros(len(feature_names), dtype=int)
    edge_importance = np.zeros(len(feature_names))

    edge_selected[top_features] = 1
    edge_importance[top_features] = rf.feature_importances_

    return {
        "accuracy":        accuracy_score(y_test, pred),
        "roc_auc":         roc_auc_score(y_test, prob),
        "edge_selected":   edge_selected,    # binary: was edge in top-K?
        "edge_importance": edge_importance,  # Gini importance (0 if not selected)
    }


# Run
N_BOOTSTRAPS = 1000
results = []
edge_selected_matrix   = np.zeros((N_BOOTSTRAPS, len(feature_names)), dtype=int)
edge_importance_matrix = np.zeros((N_BOOTSTRAPS, len(feature_names)))

valid_b = 0
for b in tqdm(range(N_BOOTSTRAPS), desc="Bootstraps"):
    res = run_bootstrap_with_edges(X_raw, X_z, y, feature_names,
                                   seed=b, K=30, max_features=5)
    if res is None:
        continue

    edge_selected_matrix[valid_b]   = res["edge_selected"]
    edge_importance_matrix[valid_b] = res["edge_importance"]
    results.append(res)
    valid_b += 1

# Trim to valid iterations
edge_selected_matrix   = edge_selected_matrix[:valid_b]
edge_importance_matrix = edge_importance_matrix[:valid_b]

# ── Stable edge summary ───────────────────────────────────────────────────────
selection_frequency = edge_selected_matrix.sum(axis=0)  # out of valid_b
mean_importance     = edge_importance_matrix.mean(axis=0)
std_importance      = edge_importance_matrix.std(axis=0)

# Threshold: selected in >50% of bootstrap iterations
threshold = valid_b * 0.50

stable_edges_boot = pd.DataFrame({
    "Feature_Name":      feature_names,
    "Boot_Frequency":    selection_frequency,
    "Boot_Freq_Pct":     selection_frequency / valid_b * 100,
    "Mean_Importance":   mean_importance,
    "Std_Importance":    std_importance,
}).query("Boot_Frequency > @threshold") \
  .sort_values(["Boot_Frequency", "Mean_Importance"], ascending=False)

print(stable_edges_boot.to_string(index=False))

Bootstraps: 100%|██████████| 1000/1000 [02:32<00:00,  6.56it/s]


[{'accuracy': 0.6190476190476191, 'roc_auc': 0.7067307692307692, 'edge_selected': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,